# Langfuse callback handler demo

Shows the Langfuse-callback-handler path: pass `LangfuseHook(client)` through
`config={"callbacks": [...]}` on `agent.invoke` / `ainvoke`, and every
successful ingest emits a Langfuse span carrying the per-ingest
`IngestResult` plus a Mermaid snapshot of the post-ingest graph.

**Requires** `agent_pinboard[langfuse]` and Langfuse credentials in env
(`LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST`).


## 0. Imports + Langfuse client

In [ ]:
from __future__ import annotations

import os
from typing import Any

from langchain.agents import create_agent
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langgraph.prebuilt import ToolRuntime
from langgraph.store.memory import InMemoryStore
from pydantic import BaseModel

from agent_pinboard import Entity, make_graph_tools, node, pin

try:
    from langfuse import Langfuse
    from agent_pinboard.integrations.langfuse_hook import LangfuseHook
except ImportError as exc:
    raise SystemExit(
        "This notebook needs the langfuse extra: "
        "`uv add 'agent-pinboard[langfuse]'` or `pip install 'agent-pinboard[langfuse]'`"
    ) from exc

client = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"),
)
handler = LangfuseHook(client)


## 1. Tiny domain — one tool that produces one fact per call

In [ ]:
IP = Entity(name="IP", description="ipv4/ipv6", normalizer=lambda v: str(v).lower())

class Lookup(BaseModel):
    queried: str = node(type=IP, description="queried IP")

@pin(model=Lookup)
@tool
def lookup(value: str, runtime: ToolRuntime) -> dict:
    """Mock IP lookup."""
    return {"queried": value}


## 2. Mock LLM that calls the tool once

Same shape as the other example notebooks — deterministic, no real LLM
needed.

In [ ]:
class MockChatModel(BaseChatModel):
    plan: list[dict[str, Any]] = []
    def __init__(self, plan: list[dict[str, Any]]) -> None:
        super().__init__()
        object.__setattr__(self, "plan", plan)
    @property
    def _llm_type(self) -> str: return "mock"
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        cursor = sum(1 for m in messages if isinstance(m, ToolMessage))
        if cursor < len(self.plan):
            step = self.plan[cursor]
            ai = AIMessage(content="", tool_calls=[{
                "name": step["tool"], "args": step["args"],
                "id": f"call-{cursor}", "type": "tool_call",
            }])
        else:
            ai = AIMessage(content="done.")
        return ChatResult(generations=[ChatGeneration(message=ai)])
    def bind_tools(self, tools, **_kwargs): return self


## 3. Run the agent with `LangfuseHook` attached

Same `create_agent` setup as `agent_demo.ipynb`. The handler is registered
in `config["callbacks"]` exactly like any other LangChain callback. After
the run, `client.flush()` makes sure all buffered spans actually leave the
process before the notebook ends.

In [ ]:
PLAN = [
    {"tool": "lookup", "args": {"value": "8.8.8.8"}},
]

agent = create_agent(
    MockChatModel(plan=PLAN),
    [lookup, *make_graph_tools()],
    store=InMemoryStore(),
)
agent.invoke(
    {"messages": [{"role": "user", "content": "Investigate 8.8.8.8."}]},
    config={
        "configurable": {"thread_id": "langfuse-demo"},
        "callbacks": [handler],
    },
)
client.flush()
print("Check your Langfuse dashboard — you should see one\n"
      "`agent_pinboard.ingest` span and one `agent_pinboard.graph_snapshot` span.")
